# Middle East — Natural Resources Atlas

Country-by-country multilayer analysis, starting with **Saudi Arabia**.

### Data Sources
- **World Bank Indicators API v2** — resource rents (oil, gas, mineral, coal, forest) as % of GDP
- **Natural Earth 50m** — country boundary geometries
- **Curated geolocated dataset** — ~45 resource sites from Wikipedia, USGS, Ma'aden, Aramco public data

### Resource Categories
| Color | Type | Examples |
|-------|------|----------|
| 🟡 Gold | Oil Fields | Ghawar, Shaybah, Safaniyah, Khurais |
| 🔵 Steel Blue | Gas Fields | Karan, Hasbah, Arabiyah |
| 🟤 Copper | Mineral Mines | Mahd Ad Dhahab (gold), Jabal Sayid (copper), Al-Jalamid (phosphate) |
| ☀️ Yellow | Solar Farms | Sudair, Sakaka, NEOM |
| 💧 Sky Blue | Desalination | Ras Al-Khair, Jubail, Shuaiba |
| ⚙️ Gray | Refineries | Ras Tanura, Yanbu, Jubail |


In [1]:
# ── Cell 2: Fetch World Bank resource-rent indicators ─────────────────────────
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

COUNTRIES = {
    "SAU": "Saudi Arabia", "IRQ": "Iraq", "IRN": "Iran",
    "ARE": "UAE", "KWT": "Kuwait", "QAT": "Qatar",
    "OMN": "Oman", "BHR": "Bahrain",
}
COUNTRY_CODES = ";".join(COUNTRIES.keys())

INDICATORS = {
    "NY.GDP.PETR.RT.ZS": "Oil rents (% of GDP)",
    "NY.GDP.NGAS.RT.ZS": "Natural gas rents (% of GDP)",
    "NY.GDP.MINR.RT.ZS": "Mineral rents (% of GDP)",
    "NY.GDP.COAL.RT.ZS": "Coal rents (% of GDP)",
    "NY.GDP.FRST.RT.ZS": "Forest rents (% of GDP)",
    "NY.GDP.TOTL.RT.ZS": "Total resource rents (% of GDP)",
    "NY.GDP.MKTP.CD":    "GDP (current US$)",
    "SP.POP.TOTL":       "Population, total",
}

def fetch_wb(indicator, countries=COUNTRY_CODES, date="2000:2023"):
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"
    all_rows, page = [], 1
    while True:
        params = {"format": "json", "per_page": 500, "date": date, "page": page}
        r = requests.get(url, params=params, timeout=30)
        raw = r.json()
        if len(raw) < 2 or raw[1] is None:
            break
        for rec in raw[1]:
            if rec["value"] is not None:
                all_rows.append({
                    "country": rec["country"]["value"],
                    "iso3": rec["countryiso3code"],
                    "year": int(rec["date"]),
                    "value": rec["value"],
                })
        if page >= raw[0]["pages"]:
            break
        page += 1
    return all_rows

print("Fetching World Bank indicators …")
all_data = {}
for code, label in INDICATORS.items():
    rows = fetch_wb(code)
    all_data[code] = pd.DataFrame(rows)
    n = len(rows)
    status = "✓" if n > 0 else "✗ (no data)"
    print(f"  {status} {code:<25s} → {n:>4d} rows  │ {label}")

# ── Latest snapshot summary ──
def latest_value(df, iso3):
    if df.empty or "iso3" not in df.columns:
        return None, None
    sub = df[df["iso3"] == iso3].sort_values("year", ascending=False)
    if sub.empty:
        return None, None
    row = sub.iloc[0]
    return row["value"], int(row["year"])

summary_rows = []
for iso3, name in COUNTRIES.items():
    row = {"Country": name, "ISO3": iso3}
    for code, label in INDICATORS.items():
        df = all_data[code]
        val, yr = latest_value(df, iso3)
        key = f"{label} ({yr})" if val is not None else label
        row[key] = val
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index("Country")
print("\nAll indicators fetched ✓\n")
df_summary.T


Fetching World Bank indicators …
  ✓ NY.GDP.PETR.RT.ZS         →  175 rows  │ Oil rents (% of GDP)
  ✓ NY.GDP.NGAS.RT.ZS         →  175 rows  │ Natural gas rents (% of GDP)
  ✓ NY.GDP.MINR.RT.ZS         →  175 rows  │ Mineral rents (% of GDP)
  ✓ NY.GDP.COAL.RT.ZS         →  175 rows  │ Coal rents (% of GDP)
  ✓ NY.GDP.FRST.RT.ZS         →  175 rows  │ Forest rents (% of GDP)
  ✓ NY.GDP.TOTL.RT.ZS         →  175 rows  │ Total resource rents (% of GDP)
  ✓ NY.GDP.MKTP.CD            →  192 rows  │ GDP (current US$)
  ✓ SP.POP.TOTL               →  192 rows  │ Population, total

All indicators fetched ✓



Country,Saudi Arabia,Iraq,Iran,UAE,Kuwait,Qatar,Oman,Bahrain
ISO3,SAU,IRQ,IRN,ARE,KWT,QAT,OMN,BHR
Oil rents (% of GDP) (2021),23.686294,42.787098,18.265898,15.673095,NaN,15.278467,23.538744,10.937676
Natural gas rents (% of GDP) (2021),1.715848,0.655523,8.808886,1.960895,NaN,12.011255,5.67148,5.701911
Mineral rents (% of GDP) (2021),0.162202,0.0,3.35588,0.0,NaN,0.0,0.0,0.0
Coal rents (% of GDP) (2021),0.0,0.0,0.009395,0.0,NaN,0.0,0.0,0.0
Forest rents (% of GDP) (2021),0.000897,0.003236,0.008001,0.000106,NaN,0.000066,0.001355,0.000448
Total resource rents (% of GDP) (2021),25.565242,43.445858,30.44806,17.634096,NaN,27.289787,29.211579,16.640034
GDP (current US$) (2023),1218584533333.330078,268881051643.549011,457510482317.482971,522622191967.325012,165384407116.233002,217308516483.515991,106174708036.509995,46192260638.297897
"Population, total (2023)",33702731,45074049,90608707,10483751,4853420,2656032,5049269,1577059
Oil rents (% of GDP) (2020),NaN,NaN,NaN,NaN,27.581689,NaN,NaN,NaN


In [2]:
# ── Cell 3: Geolocated resource sites — Saudi Arabia ─────────────────────────
# Sources: Wikipedia, USGS, Ma'aden, Saudi Aramco public filings, Saudipedia

resource_sites = [
    # ── OIL FIELDS ──────────────────────────────────────────────────────────
    {"name": "Ghawar",        "lat": 25.43,  "lon": 49.62,  "type": "Oil", "subtype": "Onshore", "notes": "World's largest conventional oil field, ~280 km long"},
    {"name": "Shaybah",       "lat": 22.51,  "lon": 53.95,  "type": "Oil", "subtype": "Onshore", "notes": "Located in the Rub' al Khali (Empty Quarter)"},
    {"name": "Safaniyah",     "lat": 28.28,  "lon": 48.75,  "type": "Oil", "subtype": "Offshore", "notes": "World's largest offshore oil field"},
    {"name": "Khurais",       "lat": 25.07,  "lon": 48.20,  "type": "Oil", "subtype": "Onshore", "notes": "Khurais–Abu Jifan–Mazalij complex, ~150 km E of Riyadh"},
    {"name": "Abqaiq",        "lat": 25.93,  "lon": 49.67,  "type": "Oil", "subtype": "Onshore", "notes": "Major processing facility & field"},
    {"name": "Berri",         "lat": 27.00,  "lon": 49.75,  "type": "Oil", "subtype": "Offshore", "notes": "Coastal/offshore field, Eastern Province"},
    {"name": "Manifa",        "lat": 27.28,  "lon": 48.93,  "type": "Oil", "subtype": "Offshore", "notes": "Shallow-water mega-field in Persian Gulf"},
    {"name": "Zuluf",         "lat": 28.00,  "lon": 48.85,  "type": "Oil", "subtype": "Offshore", "notes": "Offshore field in northern Gulf waters"},
    {"name": "Marjan",        "lat": 28.25,  "lon": 49.25,  "type": "Oil", "subtype": "Offshore", "notes": "Offshore field, major expansion 2020s"},
    {"name": "Haradh",        "lat": 24.13,  "lon": 49.07,  "type": "Oil", "subtype": "Onshore", "notes": "Southern extension of Ghawar complex"},
    {"name": "Khursaniyah",   "lat": 27.08,  "lon": 49.00,  "type": "Oil", "subtype": "Onshore", "notes": "Light crude & gas processing"},
    {"name": "Qatif",         "lat": 26.58,  "lon": 50.00,  "type": "Oil", "subtype": "Onshore", "notes": "Eastern Province coastal field"},
    {"name": "Abu Sa'fah",    "lat": 27.25,  "lon": 50.17,  "type": "Oil", "subtype": "Offshore", "notes": "Shared production zone with Bahrain"},

    # ── GAS FIELDS ──────────────────────────────────────────────────────────
    {"name": "Karan",         "lat": 27.80,  "lon": 49.50,  "type": "Gas", "subtype": "Offshore", "notes": "First non-associated offshore gas field developed"},
    {"name": "Hasbah",        "lat": 27.50,  "lon": 49.80,  "type": "Gas", "subtype": "Offshore", "notes": "Part of Wasit Gas Program"},
    {"name": "Arabiyah",      "lat": 27.60,  "lon": 49.90,  "type": "Gas", "subtype": "Offshore", "notes": "Part of Wasit Gas Program"},
    {"name": "Midyan",        "lat": 28.35,  "lon": 36.50,  "type": "Gas", "subtype": "Onshore", "notes": "Northwest Saudi Arabia, Tabuk Province"},
    {"name": "Jafurah",       "lat": 24.50,  "lon": 49.80,  "type": "Gas", "subtype": "Onshore", "notes": "Massive unconventional gas basin, under development"},

    # ── MINERAL MINES ───────────────────────────────────────────────────────
    {"name": "Mahd Ad Dhahab",  "lat": 23.50,  "lon": 40.87,  "type": "Mineral", "subtype": "Gold", "notes": "Historic gold mine, Arabian Shield"},
    {"name": "Ad Duwayhi",      "lat": 22.75,  "lon": 42.25,  "type": "Mineral", "subtype": "Gold", "notes": "Ma'aden operated, high-grade gold"},
    {"name": "Bulghah",         "lat": 24.30,  "lon": 41.40,  "type": "Mineral", "subtype": "Gold", "notes": "Open-pit gold mine, Ma'aden"},
    {"name": "Al Sukhaybarat",  "lat": 25.45,  "lon": 41.50,  "type": "Mineral", "subtype": "Gold", "notes": "Underground gold mine"},
    {"name": "Al Amar",         "lat": 22.85,  "lon": 44.55,  "type": "Mineral", "subtype": "Gold", "notes": "Underground gold-copper mine"},
    {"name": "Jabal Sayid",     "lat": 23.85,  "lon": 40.94,  "type": "Mineral", "subtype": "Copper", "notes": "Major copper mine, Al Madinah Province"},
    {"name": "Al Masane",       "lat": 19.00,  "lon": 43.50,  "type": "Mineral", "subtype": "Copper/Zinc", "notes": "Copper-zinc-gold, Najran Province"},
    {"name": "Al-Jalamid",      "lat": 31.50,  "lon": 39.75,  "type": "Mineral", "subtype": "Phosphate", "notes": "Hazm Al-Jalamid, Ma'aden phosphate complex"},
    {"name": "Umm Wu'al",       "lat": 31.20,  "lon": 37.25,  "type": "Mineral", "subtype": "Phosphate", "notes": "Second phosphate project, Northern Border"},
    {"name": "Al Ba'itha",      "lat": 26.50,  "lon": 43.20,  "type": "Mineral", "subtype": "Bauxite", "notes": "Bauxite mine, Qassim Province"},

    # ── SOLAR FARMS ─────────────────────────────────────────────────────────
    {"name": "Sudair Solar PV", "lat": 25.98,  "lon": 45.52,  "type": "Solar", "subtype": "PV", "notes": "1.5 GW, Sudair Industrial City"},
    {"name": "Sakaka Solar",    "lat": 29.95,  "lon": 40.20,  "type": "Solar", "subtype": "PV", "notes": "300 MW, Al Jouf Province"},
    {"name": "NEOM Green H2",   "lat": 28.13,  "lon": 34.92,  "type": "Solar", "subtype": "PV + Wind", "notes": "NEOM mega-project, green hydrogen"},
    {"name": "Al Shuaibah Solar","lat": 21.30, "lon": 39.10,  "type": "Solar", "subtype": "PV", "notes": "2.6 GW, near Jeddah"},
    {"name": "Ar Rass Solar",   "lat": 25.87,  "lon": 43.50,  "type": "Solar", "subtype": "PV", "notes": "700 MW, Qassim Province"},

    # ── DESALINATION ────────────────────────────────────────────────────────
    {"name": "Ras Al-Khair",    "lat": 27.54,  "lon": 49.14,  "type": "Desalination", "subtype": "MSF + RO", "notes": "World's largest desalination plant"},
    {"name": "Jubail Desal",    "lat": 27.00,  "lon": 49.65,  "type": "Desalination", "subtype": "MSF", "notes": "Jubail Industrial City complex"},
    {"name": "Shuaiba Desal",   "lat": 20.67,  "lon": 39.53,  "type": "Desalination", "subtype": "MSF + RO", "notes": "Red Sea coast, south of Jeddah"},
    {"name": "Yanbu Desal",     "lat": 24.05,  "lon": 38.00,  "type": "Desalination", "subtype": "RO", "notes": "Red Sea coast desalination"},

    # ── REFINERIES ──────────────────────────────────────────────────────────
    {"name": "Ras Tanura Refinery", "lat": 26.69,  "lon": 50.10,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "Major export terminal & refinery"},
    {"name": "Yanbu Refinery",      "lat": 23.92,  "lon": 38.28,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "Red Sea coast refinery complex"},
    {"name": "Jubail Refinery",     "lat": 27.01,  "lon": 49.66,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "SATORP JV with Total"},
    {"name": "Riyadh Refinery",     "lat": 24.60,  "lon": 46.75,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "Inland refinery for domestic supply"},
    {"name": "Jazan Refinery",      "lat": 16.90,  "lon": 42.55,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "400 kbd, newest Aramco refinery"},
    {"name": "Ras Al-Khair Smelter","lat": 27.55,  "lon": 49.15,  "type": "Refinery", "subtype": "Aluminium Smelter", "notes": "Ma'aden aluminium smelting complex"},
]

df_resources = pd.DataFrame(resource_sites)

# Summary
print(f"Total resource sites: {len(df_resources)}\n")
print(df_resources.groupby("type").size().to_string())
print(f"\nSubtypes:")
print(df_resources.groupby(["type", "subtype"]).size().to_string())


Total resource sites: 43

type
Desalination     4
Gas              5
Mineral         10
Oil             13
Refinery         6
Solar            5

Subtypes:
type          subtype          
Desalination  MSF                  1
              MSF + RO             2
              RO                   1
Gas           Offshore             3
              Onshore              2
Mineral       Bauxite              1
              Copper               1
              Copper/Zinc          1
              Gold                 5
              Phosphate            2
Oil           Offshore             6
              Onshore              7
Refinery      Aluminium Smelter    1
              Oil Refinery         5
Solar         PV                   4
              PV + Wind            1


In [3]:
# ── Cell 4: Interactive Resource Map (Folium — no API key needed) ─────────────
import folium
from folium.plugins import MarkerCluster
import geopandas as gpd
import json

# ── Load country boundary ──
NE_URL = "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip"
print("Downloading Natural Earth 50m boundaries …")
world = gpd.read_file(NE_URL)

ME_NAMES = ["Saudi Arabia", "Iraq", "Iran", "United Arab Emirates",
            "Kuwait", "Qatar", "Oman", "Bahrain", "Yemen", "Jordan",
            "Israel", "Syria", "Lebanon", "Turkey", "Egypt"]
gdf = world[world["ADMIN"].isin(ME_NAMES)][["ADMIN", "ISO_A3", "geometry"]].copy()
gdf = gdf.to_crs(epsg=4326)
print(f"Loaded {len(gdf)} country boundaries")

# ── Color map (hex) ──
COLOR_MAP = {
    "Oil":          "#DAA520",   # Gold/Amber
    "Gas":          "#4682B4",   # Steel Blue
    "Mineral":      "#B87333",   # Copper
    "Solar":        "#FFC800",   # Bright Yellow
    "Desalination": "#00BFFF",   # Deep Sky Blue
    "Refinery":     "#A9A9A9",   # Gray
}

ICON_MAP = {
    "Oil":          "tint",
    "Gas":          "fire",
    "Mineral":      "gem",
    "Solar":        "sun-o",
    "Desalination": "tint",
    "Refinery":     "industry",
}

# ── Build Folium map ──
m = folium.Map(
    location=[24.5, 45.0],
    zoom_start=5,
    tiles="CartoDB dark_matter",
    attr="CartoDB",
)

# Add country boundaries
saudi_gdf = gdf[gdf["ADMIN"] == "Saudi Arabia"]
neighbors_gdf = gdf[gdf["ADMIN"] != "Saudi Arabia"]

# Neighbors — subtle
folium.GeoJson(
    neighbors_gdf.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "#333333",
        "color": "#555555",
        "weight": 1,
        "fillOpacity": 0.15,
    },
    name="Neighbors",
).add_to(m)

# Saudi Arabia — highlighted with gold border
folium.GeoJson(
    saudi_gdf.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "#2d2823",
        "color": "#DAA520",
        "weight": 2.5,
        "fillOpacity": 0.25,
    },
    name="Saudi Arabia",
).add_to(m)

# ── Add resource markers ──
for _, row in df_resources.iterrows():
    color = COLOR_MAP.get(row["type"], "#FFFFFF")
    icon_name = ICON_MAP.get(row["type"], "info-sign")

    popup_html = (
        f"<div style='font-family:Inter,sans-serif;min-width:200px;'>"
        f"<b style='font-size:14px;color:{color};'>{row['name']}</b><br/>"
        f"<span style='color:#888;'>Type:</span> {row['type']} — {row['subtype']}<br/>"
        f"<span style='color:#888;'>Notes:</span> {row['notes']}"
        f"</div>"
    )

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=2,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row['name']} ({row['type']})",
    ).add_to(m)

# ── Add legend ──
legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:rgba(20,20,35,0.92); padding:14px 18px; border-radius:8px;
     border:1px solid #DAA520; font-family:Inter,sans-serif; color:#e0e0e0;
     font-size:12px; line-height:1.8; box-shadow:0 4px 16px rgba(0,0,0,0.5);">
  <b style="font-size:13px; color:#DAA520;">⬡ Resource Types</b><br/>
  <span style="color:#DAA520;">●</span> Oil Fields (13)<br/>
  <span style="color:#4682B4;">●</span> Gas Fields (5)<br/>
  <span style="color:#B87333;">●</span> Mineral Mines (10)<br/>
  <span style="color:#FFC800;">●</span> Solar Farms (5)<br/>
  <span style="color:#00BFFF;">●</span> Desalination (4)<br/>
  <span style="color:#A9A9A9;">●</span> Refineries (6)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Add layer control
folium.LayerControl().add_to(m)

# Save
output_path = "output/middle_east_resources.html"
m.save(output_path)
print(f"✓ Map saved to {output_path}")
print(f"  {len(df_resources)} resource sites plotted")
print(f"  {len(gdf)} country boundaries rendered")


Loaded 15 country boundaries
✓ Map saved to output/middle_east_resources.html
  43 resource sites plotted
  15 country boundaries rendered


In [ ]:
# ── Cell 5: 3D Production Capacity Map (PyDeck PolygonLayer) ───────────────────
import pydeck as pdk
import pandas as pd
import math
from pathlib import Path

BAR_ELEVATION_SCALE = 3000
LABEL_Z_OFFSET = 20000

# Add a normalized capacity scale (1-100) to represent relative production size.
# For example, Ghawar is massive (100), while smaller fields/plants get lower values.
capacity_mapping = {
    "Ghawar": 100, "Safaniyah": 80, "Khurais": 60, "Shaybah": 50, "Manifa": 50, "Zuluf": 45, "Marjan": 45,
    "Abqaiq": 40, "Berri": 35, "Haradh": 30, "Khursaniyah": 30, "Qatif": 25, "Abu Sa'fah": 25,
    "Karan": 40, "Hasbah": 35, "Arabiyah": 35, "Jafurah": 50, "Midyan": 15,
    "Mahd Ad Dhahab": 20, "Jabal Sayid": 25, "Al-Jalamid": 35, "Umm Wu'al": 30, "Al Ba'itha": 30,
    "Ad Duwayhi": 20, "Bulghah": 15, "Al Sukhaybarat": 15, "Al Amar": 15, "Al Masane": 15,
    "Sudair Solar PV": 35, "Sakaka Solar": 15, "NEOM Green H2": 45, "Al Shuaibah Solar": 40, "Ar Rass Solar": 25,
    "Ras Al-Khair": 50, "Jubail Desal": 40, "Shuaiba Desal": 35, "Yanbu Desal": 25,
    "Ras Tanura Refinery": 60, "Yanbu Refinery": 45, "Jubail Refinery": 45, "Jazan Refinery": 40,
    "Riyadh Refinery": 25, "Ras Al-Khair Smelter": 35
}

metric_mapping = {
    "Ghawar": "Oil reserves: >48 billion bbl",
    "Shaybah": "Oil capacity: 1.0 million bbl/d",
    "Safaniyah": "Oil capacity: 1.3 million bbl/d",
    "Khurais": "Oil capacity: 1.45 million bbl/d",
    "Abqaiq": "Processing capacity: 7 million bbl/d",
    "Berri": "Oil capacity: 250,000 bbl/d",
    "Manifa": "Oil capacity: 900,000 bbl/d",
    "Zuluf": "Oil capacity target: 600,000 bbl/d",
    "Marjan": "Oil capacity target: 800,000 bbl/d",
    "Haradh": "Oil capacity: 300,000 bbl/d",
    "Khursaniyah": "Oil capacity: 500,000 bbl/d",
    "Qatif": "Oil capacity: 500,000 bbl/d",
    "Abu Sa'fah": "Oil capacity: 300,000 bbl/d",
    "Karan": "Gas capacity: 1.8 Bcf/d",
    "Hasbah": "Gas feed: 1.3 Bcf/d",
    "Arabiyah": "Gas feed: 1.2 Bcf/d",
    "Midyan": "Gas capacity: 75 MMcf/d",
    "Jafurah": "Gas target: 2.0 Bcf/d",
    "Mahd Ad Dhahab": "Gold output: 30,000 oz/yr",
    "Ad Duwayhi": "Gold output: 180,000 oz/yr",
    "Bulghah": "Gold output: 60,000 oz/yr",
    "Al Sukhaybarat": "Gold output: 30,000 oz/yr",
    "Al Amar": "Gold output: 45,000 oz/yr",
    "Jabal Sayid": "Copper output: 60,000 t/yr",
    "Al Masane": "Ore throughput: 700,000 t/yr",
    "Al-Jalamid": "Phosphate rock: 11 million t/yr",
    "Umm Wu'al": "Phosphate product: 3 million t/yr",
    "Al Ba'itha": "Bauxite output: 4 million t/yr",
    "Sudair Solar PV": "Solar capacity: 1,500 MW",
    "Sakaka Solar": "Solar capacity: 300 MW",
    "NEOM Green H2": "Renewables feed: 4 GW",
    "Al Shuaibah Solar": "Solar capacity: 2,600 MW",
    "Ar Rass Solar": "Solar capacity: 700 MW",
    "Ras Al-Khair": "Water capacity: 1,050,000 m3/day",
    "Jubail Desal": "Water capacity: 800,000 m3/day",
    "Shuaiba Desal": "Water capacity: 880,000 m3/day",
    "Yanbu Desal": "Water capacity: 550,000 m3/day",
    "Ras Tanura Refinery": "Refining capacity: 550,000 bbl/d",
    "Yanbu Refinery": "Refining capacity: 400,000 bbl/d",
    "Jubail Refinery": "Refining capacity: 440,000 bbl/d",
    "Riyadh Refinery": "Refining capacity: 126,000 bbl/d",
    "Jazan Refinery": "Refining capacity: 400,000 bbl/d",
    "Ras Al-Khair Smelter": "Aluminium output: 740,000 t/yr",
    "Rumaila": "Oil capacity: 1.4 million bbl/d",
    "West Qurna": "Oil capacity: 550,000 bbl/d",
    "Majnoon": "Oil capacity: 450,000 bbl/d",
    "Kirkuk": "Oil reserves: 8.7 billion bbl",
    "Zubair": "Oil capacity: 475,000 bbl/d",
    "Halfaya": "Oil capacity: 400,000 bbl/d",
    "Akkas": "Gas reserves: 5.6 Tcf",
    "Mansuriya": "Gas reserves: 4.5 Tcf",
    "Baiji Refinery": "Refining capacity: 310,000 bbl/d",
    "Karbala Refinery": "Refining capacity: 140,000 bbl/d",
    "South Pars": "Gas reserves: 1,200 Tcf",
    "Ahvaz": "Oil reserves: 65 billion bbl",
    "Marun": "Oil reserves: 22 billion bbl",
    "Gachsaran": "Oil reserves: 53 billion bbl",
    "Azadegan": "Oil reserves: 32 billion bbl",
    "Yadavaran": "Oil reserves: 12 billion bbl",
    "Sarcheshmeh": "Copper output: 280,000 t/yr",
    "Chadormalu": "Iron ore output: 10 million t/yr",
    "Abadan Refinery": "Refining capacity: 400,000 bbl/d",
    "Esfahan Refinery": "Refining capacity: 375,000 bbl/d",
    "Upper Zakum": "Oil capacity: 750,000 bbl/d",
    "Lower Zakum": "Oil capacity: 450,000 bbl/d",
    "Bab": "Oil capacity: 420,000 bbl/d",
    "Bu Hasa": "Oil capacity: 650,000 bbl/d",
    "Shah Gas": "Gas capacity: 1.0 Bcf/d",
    "Ruwais Refinery": "Refining capacity: 837,000 bbl/d",
    "Jebel Ali Desalination": "Water capacity: 2.2 million m3/day",
    "Taweelah Desalination": "Water capacity: 909,000 m3/day",
    "Al Dhafra Solar": "Solar capacity: 2,000 MW",
    "MBR Solar Park": "Solar target: 5,000 MW",
    "Burgan": "Oil reserves: 66-70 billion bbl",
    "Raudhatain": "Oil capacity: 230,000 bbl/d",
    "Sabriya": "Oil capacity: 120,000 bbl/d",
    "Minagish": "Oil capacity: 190,000 bbl/d",
    "Umm Gudair": "Oil capacity: 110,000 bbl/d",
    "Jurassic Gas": "Gas capacity: 600 MMcf/d",
    "Mina Al Ahmadi Refinery": "Refining capacity: 466,000 bbl/d",
    "Al Zour Refinery": "Refining capacity: 615,000 bbl/d",
    "Doha Desalination": "Water capacity: 250,000 m3/day",
    "Shagaya Renewables": "Renewable capacity: 70 MW",
    "North Field": "Gas reserves: 900 Tcf",
    "Dukhan": "Oil capacity: 300,000 bbl/d",
    "Al Shaheen": "Oil capacity: 300,000 bbl/d",
    "Idd El-Shargi": "Oil capacity: 100,000 bbl/d",
    "Bul Hanine": "Oil capacity: 45,000 bbl/d",
    "Pearl GTL": "GTL output: 140,000 bbl/d",
    "Ras Laffan Refinery": "Refining capacity: 292,000 bbl/d",
    "Ras Abu Fontas Desal": "Water capacity: 160,000 m3/day",
    "Al Kharsaah Solar": "Solar capacity: 800 MW",
    "Fahud": "Oil capacity: 100,000 bbl/d",
    "Yibal": "Oil capacity: 40,000 bbl/d",
    "Marmul": "Oil capacity: 80,000 bbl/d",
    "Mukhaizna": "Oil capacity: 120,000 bbl/d",
    "Khazzan": "Gas capacity: 1.5 Bcf/d",
    "Saih Rawl": "Gas capacity: 500 MMcf/d",
    "Sohar Refinery": "Refining capacity: 198,000 bbl/d",
    "Mina Al Fahal Refinery": "Refining capacity: 106,000 bbl/d",
    "Ibri Solar": "Solar capacity: 500 MW",
    "Sohar Aluminium": "Aluminium output: 395,000 t/yr",
    "Bahrain Field": "Oil capacity: 45,000 bbl/d",
    "Khaleej Al Bahrain": "Resource estimate: 80 billion bbl oil in place",
    "Tatweer Gas": "Gas output: 1.5 Bcf/d",
    "Sitra Refinery": "Refining capacity: 267,000 bbl/d",
    "Alba Smelter": "Aluminium output: 1.6 million t/yr",
    "Al Dur Desalination": "Water capacity: 218,000 m3/day",
    "Hidd Desalination": "Water capacity: 409,000 m3/day",
    "Askar Solar": "Solar capacity: 5 MW",
    "Masila Block 14": "Oil capacity: 120,000 bbl/d peak",
    "Marib-Jawf Basin": "Oil capacity: 200,000 bbl/d peak",
    "Shabwa Block S1": "Oil capacity: 50,000 bbl/d peak",
    "Jannah Hunt": "Oil capacity: 45,000 bbl/d peak",
    "Marib Gas": "Gas reserves: 10 Tcf",
    "Balhaf LNG": "LNG capacity: 6.7 million t/yr",
    "Aden Refinery": "Refining capacity: 150,000 bbl/d",
    "Marib Refinery": "Refining capacity: 10,000 bbl/d",
    "Al Jawf Solar": "Solar resource: 2,200 kWh/m2/yr",
    "Risha Gas": "Gas output: 16 MMcf/d",
    "Hamza Oil Field": "Oil output: 1,000 bbl/d",
    "Arab Potash": "Potash output: 2.5 million t/yr",
    "Jordan Bromine": "Bromine output: 140,000 t/yr",
    "Eshidiya Phosphate": "Phosphate output: 6 million t/yr",
    "Al-Abiad Phosphate": "Phosphate output: 1 million t/yr",
    "Attarat Oil Shale": "Power capacity: 470 MW",
    "Ma an Solar": "Solar capacity: 160 MW",
    "Quweira Solar": "Solar capacity: 103 MW",
    "Leviathan": "Gas reserves: 22 Tcf",
    "Tamar": "Gas reserves: 10 Tcf",
    "Karish": "Gas reserves: 1.4 Tcf",
    "Tanin": "Gas reserves: 1.2 Tcf",
    "Dead Sea Works": "Potash output: 4 million t/yr",
    "Haifa Refinery": "Refining capacity: 197,000 bbl/d",
    "Ashdod Refinery": "Refining capacity: 100,000 bbl/d",
    "Sorek Desalination": "Water capacity: 624,000 m3/day",
    "Hadera Desalination": "Water capacity: 525,000 m3/day",
    "Ashalim Solar": "Solar capacity: 250 MW",
    "Al-Omar": "Oil capacity: 80,000 bbl/d peak",
    "Al-Tanak": "Oil capacity: 40,000 bbl/d peak",
    "Rmelan": "Oil capacity: 90,000 bbl/d peak",
    "Jbessa": "Oil capacity: 35,000 bbl/d peak",
    "Al-Shaer Gas": "Gas output: 3 million m3/day peak",
    "Conoco Gas Plant": "Gas processing: 450 MMcf/d",
    "Banias Refinery": "Refining capacity: 130,000 bbl/d",
    "Homs Refinery": "Refining capacity: 107,000 bbl/d",
    "Palmyra Phosphate": "Phosphate output: 2 million t/yr",
    "Khnifis Phosphate": "Phosphate output: 1 million t/yr",
    "Offshore Block 9": "Exploration block: no commercial output yet",
    "Offshore Block 4": "Exploration block: no commercial output yet",
    "Tripoli Oil Installations": "Oil storage: 250,000 m3",
    "Zahrani Oil Terminal": "Oil storage: 180,000 m3",
    "Sibline Limestone": "Cement capacity: 1.2 million t/yr",
    "Chekka Limestone": "Cement capacity: 2 million t/yr",
    "Beirut River Solar": "Solar capacity: 1 MW",
    "Zahrani Solar": "Solar capacity: 12 MW",
    "Sakarya Gas Field": "Gas reserves: 710 Bcm",
    "Raman": "Oil output: 7,000 bbl/d",
    "Garzan": "Oil output: 5,000 bbl/d",
    "Tuz Golu Gas Storage": "Storage capacity: 1.2 Bcm",
    "Izmit Refinery": "Refining capacity: 227,000 bbl/d",
    "Izmir Refinery": "Refining capacity: 239,000 bbl/d",
    "STAR Refinery": "Refining capacity: 214,000 bbl/d",
    "Kirka Boron": "Boron product: 2.5 million t/yr",
    "Bigadic Boron": "Boron product: 1 million t/yr",
    "Karapinar Solar": "Solar capacity: 1,350 MW",
    "Zohr": "Gas reserves: 30 Tcf",
    "West Delta Deep Marine": "Gas output: 1.5 Bcf/d",
    "Abu Madi": "Gas reserves: 10 Tcf",
    "Belayim": "Oil output: 90,000 bbl/d",
    "Ras Gharib": "Oil output: 25,000 bbl/d",
    "Sukari Gold Mine": "Gold output: 450,000 oz/yr",
    "Abu Tartur Phosphate": "Phosphate reserves: 1 billion t",
    "Benban Solar Park": "Solar capacity: 1,650 MW",
    "Suez Refinery": "Refining capacity: 68,000 bbl/d",
    "MIDOR Refinery": "Refining capacity: 160,000 bbl/d",
    "Al Galala Desalination": "Water capacity: 150,000 m3/day"
}

# Build a JSON-serializable copy for PyDeck. Passing the pandas DataFrame
# directly can serialize as an empty internal manager object in newer pandas.
df_3d = df_resources.copy()
df_3d['capacity_scale'] = df_3d['name'].map(capacity_mapping).fillna(10).astype(float)
df_3d['metric_text'] = df_3d['name'].map(metric_mapping).fillna('Metric unavailable')
df_3d['lat'] = pd.to_numeric(df_3d['lat'], errors='raise').astype(float)
df_3d['lon'] = pd.to_numeric(df_3d['lon'], errors='raise').astype(float)
df_3d['label_z'] = df_3d['capacity_scale'] * BAR_ELEVATION_SCALE + LABEL_Z_OFFSET
df_3d['label_text'] = df_3d.apply(lambda row: f"{row['name']}\n{row['metric_text']}", axis=1)

# Map colors to RGB lists for PyDeck
COLOR_MAP_RGB = {
    "Oil": [255, 218, 64, 95],
    "Gas": [72, 210, 255, 95],
    "Mineral": [255, 132, 72, 95],
    "Solar": [255, 245, 84, 95],
    "Desalination": [0, 235, 255, 95],
    "Refinery": [220, 226, 238, 95],
}

RESOURCE_METRIC_HINTS = {
    "Oil": "barrels per day or barrels of reserves",
    "Gas": "billion/million cubic feet per day; trillion cubic feet; billion cubic meters",
    "Mineral": "tonnes per year or troy ounces per year",
    "Solar": "megawatts or gigawatts",
    "Desalination": "cubic meters per day",
    "Refinery": "barrels per day or tonnes per year",
}

SCATTER_GROUP_SIZE_DEGREES = 0.1
SCATTER_RADIUS_DEGREES = 0.06
HEX_RADIUS_DEGREES = 0.035


def rgb_to_hex(rgb):
    return f"#{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"


def build_hex_polygon(lat, lon, radius_degrees=HEX_RADIUS_DEGREES):
    lon_scale = max(math.cos(math.radians(float(lat))), 0.25)
    return [
        [float(lon) + (radius_degrees * math.cos(angle) / lon_scale), float(lat) + radius_degrees * math.sin(angle)]
        for angle in [2 * math.pi * i / 6 for i in range(6)]
    ]


def add_scatter_offsets(df, group_size_degrees=SCATTER_GROUP_SIZE_DEGREES, radius_degrees=SCATTER_RADIUS_DEGREES):
    df = df.copy()
    df["plot_lat"] = df["lat"].astype(float)
    df["plot_lon"] = df["lon"].astype(float)
    if df.empty:
        return df

    group_keys = list(zip((df["lat"] / group_size_degrees).apply(math.floor), (df["lon"] / group_size_degrees).apply(math.floor)))
    grouped_indices = {}
    for row_index, key in zip(df.index, group_keys):
        grouped_indices.setdefault(key, []).append(row_index)

    for indices in grouped_indices.values():
        if len(indices) <= 1:
            continue
        ordered_indices = sorted(indices, key=lambda idx: (str(df.at[idx, "type"]), str(df.at[idx, "name"])))
        for offset_index, row_index in enumerate(ordered_indices):
            angle = (2 * math.pi * offset_index) / len(ordered_indices)
            lat = float(df.at[row_index, "lat"])
            lon_scale = max(math.cos(math.radians(lat)), 0.25)
            df.at[row_index, "plot_lat"] = lat + radius_degrees * math.sin(angle)
            df.at[row_index, "plot_lon"] = float(df.at[row_index, "lon"]) + (radius_degrees * math.cos(angle) / lon_scale)
    return df

    group_keys = list(zip((df["lat"] / group_size_degrees).apply(math.floor), (df["lon"] / group_size_degrees).apply(math.floor)))
    grouped_indices = {}
    for row_index, key in zip(df.index, group_keys):
        grouped_indices.setdefault(key, []).append(row_index)

    for indices in grouped_indices.values():
        if len(indices) <= 1:
            continue
        ordered_indices = sorted(indices, key=lambda idx: (str(df.at[idx, "type"]), str(df.at[idx, "name"])))
        for offset_index, row_index in enumerate(ordered_indices):
            angle = (2 * math.pi * offset_index) / len(ordered_indices)
            lat = float(df.at[row_index, "lat"])
            lon_scale = max(math.cos(math.radians(lat)), 0.25)
            df.at[row_index, "plot_lat"] = lat + radius_degrees * math.sin(angle)
            df.at[row_index, "plot_lon"] = float(df.at[row_index, "lon"]) + (radius_degrees * math.cos(angle) / lon_scale)
    return df

df_3d = add_scatter_offsets(df_3d)
df_3d['polygon'] = df_3d.apply(lambda row: build_hex_polygon(row['plot_lat'], row['plot_lon']), axis=1)
df_3d['color_rgb'] = df_3d['type'].map(COLOR_MAP_RGB)
df_3d['color_rgb'] = df_3d['color_rgb'].apply(lambda c: c if isinstance(c, list) else [255, 255, 255, 95])
column_data = df_3d[
    ['name', 'lat', 'lon', 'plot_lat', 'plot_lon', 'polygon', 'type', 'subtype', 'notes', 'metric_text', 'capacity_scale', 'label_z', 'label_text', 'color_rgb']
].to_dict('records')

print("Building 3D PolygonLayer...")

layer_3d = pdk.Layer(
    "PolygonLayer",
    column_data,
    get_polygon="polygon",
    get_elevation="capacity_scale",
    elevation_scale=BAR_ELEVATION_SCALE,  # Adjust height multiplier
    get_fill_color="color_rgb",
    extruded=True,
    pickable=True,
    auto_highlight=True,
)

label_layer = pdk.Layer(
    "TextLayer",
    column_data,
    get_position=["plot_lon", "plot_lat", "label_z"],
    get_text="label_text",
    get_size=9,
    get_color=[245, 245, 245, 230],
    get_pixel_offset=[0, -6],
    billboard=True,
    pickable=False,
)

view_state_3d = pdk.ViewState(
    latitude=24.5,
    longitude=45.0,
    zoom=5,
    pitch=45,  # Tilted to see 3D effect
    bearing=0,
)

deck_3d = pdk.Deck(
    layers=[layer_3d, label_layer],
    initial_view_state=view_state_3d,
    tooltip={
        "html": "<b>{name}</b><br>" \
                "Type: {type} - {subtype}<br>" \
                "Metric: {metric_text}<br>" \
                "Original location: {lat}, {lon}",
        "style": {"backgroundColor": "#1a1a2e", "color": "white", "border": "1px solid #daa520"}
    },
    map_style="dark" # Standard dark map
)

output_3d_path = "output/middle_east_3d_bars.html"
output_file = Path(output_3d_path)
deck_3d.to_html(output_3d_path)

legend_items = "\n".join(
    f"<div style='display:grid;grid-template-columns:10px 82px 1fr;align-items:center;gap:8px;margin:4px 0;'>"
    f"<span style='width:10px;height:10px;border-radius:2px;display:inline-block;background:{rgb_to_hex(rgb)};box-shadow:0 0 8px {rgb_to_hex(rgb)};'></span>"
    f"<span style='font-weight:700;color:#f7f8ff;'>{resource_type}</span>"
    f"<span style='color:#b9c0d4;'>{RESOURCE_METRIC_HINTS.get(resource_type, 'site metric')}</span>"
    f"</div>"
    for resource_type, rgb in COLOR_MAP_RGB.items()
)
legend_html = (
    "<div id='resource-legend' style='position:fixed;bottom:24px;left:24px;z-index:10;"
    "max-width:430px;background:rgba(8,10,20,0.88);border:1px solid rgba(72,210,255,0.62);"
    "border-radius:8px;padding:11px 13px;color:#f3f4f8;font-family:Inter,Arial,sans-serif;"
    "font-size:11px;line-height:1.25;box-shadow:0 0 22px rgba(0,235,255,0.18),0 8px 24px rgba(0,0,0,0.38);'>"
    "<div style='font-weight:800;margin-bottom:7px;color:#ffffff;'>Resource metrics</div>"
    f"{legend_items}"
    "<div style='margin-top:8px;color:#c7cad4;font-size:10px;'>Height = normalized; hex polygons are offset for colocated sites.</div>"
    "</div>"
)
html = output_file.read_text(encoding="utf-8")
if "</body>" in html:
    html = html.replace("</body>", f"{legend_html}\n  </body>", 1)
else:
    html += legend_html
output_file.write_text(html, encoding="utf-8")
print(f"✓ 3D Map saved to {output_3d_path}")


In [ ]:
# Cell 6: Shared country 3D polygon renderer (matches Saudi Arabia PyDeck style)
import pydeck as pdk
import pandas as pd
import math
from pathlib import Path

BAR_ELEVATION_SCALE = 3000
LABEL_Z_OFFSET = 20000

COLOR_MAP_RGB = {
    "Oil": [255, 218, 64, 95],
    "Gas": [72, 210, 255, 95],
    "Mineral": [255, 132, 72, 95],
    "Solar": [255, 245, 84, 95],
    "Desalination": [0, 235, 255, 95],
    "Refinery": [220, 226, 238, 95],
}

RESOURCE_METRIC_HINTS = {
    "Oil": "barrels per day or barrels of reserves",
    "Gas": "billion/million cubic feet per day; trillion cubic feet; billion cubic meters",
    "Mineral": "tonnes per year or troy ounces per year",
    "Solar": "megawatts or gigawatts",
    "Desalination": "cubic meters per day",
    "Refinery": "barrels per day or tonnes per year",
}

SCATTER_GROUP_SIZE_DEGREES = 0.1
SCATTER_RADIUS_DEGREES = 0.06
HEX_RADIUS_DEGREES = 0.035


def rgb_to_hex(rgb):
    return f"#{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"


def build_hex_polygon(lat, lon, radius_degrees=HEX_RADIUS_DEGREES):
    lon_scale = max(math.cos(math.radians(float(lat))), 0.25)
    return [
        [float(lon) + (radius_degrees * math.cos(angle) / lon_scale), float(lat) + radius_degrees * math.sin(angle)]
        for angle in [2 * math.pi * i / 6 for i in range(6)]
    ]


def add_scatter_offsets(df, group_size_degrees=SCATTER_GROUP_SIZE_DEGREES, radius_degrees=SCATTER_RADIUS_DEGREES):
    df = df.copy()
    df["plot_lat"] = df["lat"].astype(float)
    df["plot_lon"] = df["lon"].astype(float)
    if df.empty:
        return df

    group_keys = list(zip((df["lat"] / group_size_degrees).apply(math.floor), (df["lon"] / group_size_degrees).apply(math.floor)))
    grouped_indices = {}
    for row_index, key in zip(df.index, group_keys):
        grouped_indices.setdefault(key, []).append(row_index)

    for indices in grouped_indices.values():
        if len(indices) <= 1:
            continue
        ordered_indices = sorted(indices, key=lambda idx: (str(df.at[idx, "type"]), str(df.at[idx, "name"])))
        for offset_index, row_index in enumerate(ordered_indices):
            angle = (2 * math.pi * offset_index) / len(ordered_indices)
            lat = float(df.at[row_index, "lat"])
            lon_scale = max(math.cos(math.radians(lat)), 0.25)
            df.at[row_index, "plot_lat"] = lat + radius_degrees * math.sin(angle)
            df.at[row_index, "plot_lon"] = float(df.at[row_index, "lon"]) + (radius_degrees * math.cos(angle) / lon_scale)
    return df



def render_country_3d_bars(df_country, country_name, output_path, center, zoom=5, pitch=45, bearing=0):
    required_columns = {"name", "lat", "lon", "type", "subtype", "notes", "capacity_scale", "metric_text"}
    missing_columns = required_columns - set(df_country.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns for {country_name}: {sorted(missing_columns)}")

    df_3d = df_country.copy()
    df_3d["capacity_scale"] = pd.to_numeric(df_3d["capacity_scale"], errors="raise").astype(float)
    df_3d["lat"] = pd.to_numeric(df_3d["lat"], errors="raise").astype(float)
    df_3d["lon"] = pd.to_numeric(df_3d["lon"], errors="raise").astype(float)
    df_3d["metric_text"] = df_3d["metric_text"].astype(str)
    df_3d["label_z"] = df_3d["capacity_scale"] * BAR_ELEVATION_SCALE + LABEL_Z_OFFSET
    df_3d["label_text"] = df_3d.apply(
        lambda row: f"{row['name']}\n{row['metric_text']}",
        axis=1,
    )
    df_3d = add_scatter_offsets(df_3d)
    df_3d["polygon"] = df_3d.apply(lambda row: build_hex_polygon(row["plot_lat"], row["plot_lon"]), axis=1)
    df_3d["color_rgb"] = df_3d["type"].map(COLOR_MAP_RGB)
    df_3d["color_rgb"] = df_3d["color_rgb"].apply(
        lambda color: color if isinstance(color, list) else [255, 255, 255, 95]
    )

    column_data = df_3d[
        ["name", "lat", "lon", "plot_lat", "plot_lon", "polygon", "type", "subtype", "notes", "metric_text", "capacity_scale", "label_z", "label_text", "color_rgb"]
    ].to_dict("records")

    print(f"Building 3D PolygonLayer for {country_name}...")

    layer_3d = pdk.Layer(
        "PolygonLayer",
        column_data,
        get_polygon="polygon",
        get_elevation="capacity_scale",
        elevation_scale=BAR_ELEVATION_SCALE,
        get_fill_color="color_rgb",
        extruded=True,
        pickable=True,
        auto_highlight=True,
    )

    label_layer = pdk.Layer(
        "TextLayer",
        column_data,
        get_position=["plot_lon", "plot_lat", "label_z"],
        get_text="label_text",
        get_size=9,
        get_color=[245, 245, 245, 230],
        get_pixel_offset=[0, -6],
        billboard=True,
        pickable=False,
    )

    view_state_3d = pdk.ViewState(
        latitude=center[0],
        longitude=center[1],
        zoom=zoom,
        pitch=pitch,
        bearing=bearing,
    )

    deck_3d = pdk.Deck(
        layers=[layer_3d, label_layer],
        initial_view_state=view_state_3d,
        tooltip={
            "html": "<b>{name}</b><br>"
                    "Type: {type} - {subtype}<br>"
                    "Metric: {metric_text}<br>"
                    "Original location: {lat}, {lon}",
            "style": {"backgroundColor": "#1a1a2e", "color": "white", "border": "1px solid #48d2ff"},
        },
        map_style="dark",
    )

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    deck_3d.to_html(str(output_file))

    legend_items = "\n".join(
        f"<div style='display:grid;grid-template-columns:10px 82px 1fr;align-items:center;gap:8px;margin:4px 0;'>"
        f"<span style='width:10px;height:10px;border-radius:2px;display:inline-block;background:{rgb_to_hex(rgb)};box-shadow:0 0 8px {rgb_to_hex(rgb)};'></span>"
        f"<span style='font-weight:700;color:#f7f8ff;'>{resource_type}</span>"
        f"<span style='color:#b9c0d4;'>{RESOURCE_METRIC_HINTS.get(resource_type, 'site metric')}</span>"
        f"</div>"
        for resource_type, rgb in COLOR_MAP_RGB.items()
    )
    legend_html = (
        "<div id='resource-legend' style='position:fixed;bottom:24px;left:24px;z-index:10;"
        "max-width:430px;background:rgba(8,10,20,0.88);border:1px solid rgba(72,210,255,0.62);"
        "border-radius:8px;padding:11px 13px;color:#f3f4f8;font-family:Inter,Arial,sans-serif;"
        "font-size:11px;line-height:1.25;box-shadow:0 0 22px rgba(0,235,255,0.18),0 8px 24px rgba(0,0,0,0.38);'>"
        "<div style='font-weight:800;margin-bottom:7px;color:#ffffff;'>Resource metrics</div>"
        f"{legend_items}"
        "<div style='margin-top:8px;color:#c7cad4;font-size:10px;'>Height = normalized; hex polygons are offset for colocated sites.</div>"
        "</div>"
    )

    html = output_file.read_text(encoding="utf-8")
    if "</body>" in html:
        html = html.replace("</body>", f"{legend_html}\n  </body>", 1)
    else:
        html += legend_html
    output_file.write_text(html, encoding="utf-8")
    print(f"Saved 3D map to {output_path}")
    return output_file


In [ ]:
# Cell 7: Curated resource sites - Iraq
iraq_resource_sites = [
    {
        "name": "Rumaila",
        "lat": 30.48,
        "lon": 47.36,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Supergiant southern oil field near Basra",
        "capacity_scale": 95,
        "metric_text": "Oil capacity: 1.4 million bbl/d"
    },
    {
        "name": "West Qurna",
        "lat": 31.02,
        "lon": 47.43,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Large southern oil field complex",
        "capacity_scale": 90,
        "metric_text": "Oil capacity: 550,000 bbl/d"
    },
    {
        "name": "Majnoon",
        "lat": 31.17,
        "lon": 47.55,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Major field north of Basra",
        "capacity_scale": 75,
        "metric_text": "Oil capacity: 450,000 bbl/d"
    },
    {
        "name": "Kirkuk",
        "lat": 35.47,
        "lon": 44.39,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Historic giant oil field in northern Iraq",
        "capacity_scale": 70,
        "metric_text": "Oil reserves: 8.7 billion bbl"
    },
    {
        "name": "Zubair",
        "lat": 30.36,
        "lon": 47.63,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Basra-area oil field",
        "capacity_scale": 65,
        "metric_text": "Oil capacity: 475,000 bbl/d"
    },
    {
        "name": "Halfaya",
        "lat": 31.65,
        "lon": 47.31,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Maysan governorate oil field",
        "capacity_scale": 55,
        "metric_text": "Oil capacity: 400,000 bbl/d"
    },
    {
        "name": "Akkas",
        "lat": 34.04,
        "lon": 41.05,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Western Iraq gas field",
        "capacity_scale": 35,
        "metric_text": "Gas reserves: 5.6 Tcf"
    },
    {
        "name": "Mansuriya",
        "lat": 34.19,
        "lon": 44.69,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Diyala province gas field",
        "capacity_scale": 32,
        "metric_text": "Gas reserves: 4.5 Tcf"
    },
    {
        "name": "Baiji Refinery",
        "lat": 34.93,
        "lon": 43.49,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Large northern refining complex",
        "capacity_scale": 45,
        "metric_text": "Refining capacity: 310,000 bbl/d"
    },
    {
        "name": "Karbala Refinery",
        "lat": 32.46,
        "lon": 43.87,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Modern refinery serving central Iraq",
        "capacity_scale": 38,
        "metric_text": "Refining capacity: 140,000 bbl/d"
    }
]

df_resources_iraq = pd.DataFrame(iraq_resource_sites)

print(f"Total Iraq resource sites: {len(df_resources_iraq)}\n")
print(df_resources_iraq.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_iraq.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 8: 3D resource bars - Iraq
render_country_3d_bars(
    df_resources_iraq,
    country_name='Iraq',
    output_path='output/middle_east_3d_bars_iraq.html',
    center=[33.0, 43.5],
    zoom=5,
)


In [ ]:
# Cell 9: Curated resource sites - Iran
iran_resource_sites = [
    {
        "name": "South Pars",
        "lat": 27.5,
        "lon": 52.6,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Iranian side of the North Field/South Pars gas reservoir",
        "capacity_scale": 100,
        "metric_text": "Gas reserves: 1,200 Tcf"
    },
    {
        "name": "Ahvaz",
        "lat": 31.32,
        "lon": 48.68,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Giant oil field in Khuzestan",
        "capacity_scale": 85,
        "metric_text": "Oil reserves: 65 billion bbl"
    },
    {
        "name": "Marun",
        "lat": 30.75,
        "lon": 49.05,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Major southwestern oil field",
        "capacity_scale": 75,
        "metric_text": "Oil reserves: 22 billion bbl"
    },
    {
        "name": "Gachsaran",
        "lat": 30.36,
        "lon": 50.8,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Large Zagros oil field",
        "capacity_scale": 70,
        "metric_text": "Oil reserves: 53 billion bbl"
    },
    {
        "name": "Azadegan",
        "lat": 31.4,
        "lon": 47.72,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Large oil field near the Iraq border",
        "capacity_scale": 65,
        "metric_text": "Oil reserves: 32 billion bbl"
    },
    {
        "name": "Yadavaran",
        "lat": 31.02,
        "lon": 48.1,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Southwestern oil field near Abadan plain",
        "capacity_scale": 55,
        "metric_text": "Oil reserves: 12 billion bbl"
    },
    {
        "name": "Sarcheshmeh",
        "lat": 30.0,
        "lon": 55.8,
        "type": "Mineral",
        "subtype": "Copper",
        "notes": "Major copper mining and smelting complex",
        "capacity_scale": 55,
        "metric_text": "Copper output: 280,000 t/yr"
    },
    {
        "name": "Chadormalu",
        "lat": 32.31,
        "lon": 55.53,
        "type": "Mineral",
        "subtype": "Iron Ore",
        "notes": "Large central Iran iron ore mine",
        "capacity_scale": 45,
        "metric_text": "Iron ore output: 10 million t/yr"
    },
    {
        "name": "Abadan Refinery",
        "lat": 30.34,
        "lon": 48.28,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Historic refinery near the Shatt al-Arab",
        "capacity_scale": 60,
        "metric_text": "Refining capacity: 400,000 bbl/d"
    },
    {
        "name": "Esfahan Refinery",
        "lat": 32.73,
        "lon": 51.75,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Major inland refining complex",
        "capacity_scale": 55,
        "metric_text": "Refining capacity: 375,000 bbl/d"
    }
]

df_resources_iran = pd.DataFrame(iran_resource_sites)

print(f"Total Iran resource sites: {len(df_resources_iran)}\n")
print(df_resources_iran.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_iran.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 10: 3D resource bars - Iran
render_country_3d_bars(
    df_resources_iran,
    country_name='Iran',
    output_path='output/middle_east_3d_bars_iran.html',
    center=[31.8, 53.0],
    zoom=4.4,
)


In [ ]:
# Cell 11: Curated resource sites - United Arab Emirates
uae_resource_sites = [
    {
        "name": "Upper Zakum",
        "lat": 25.05,
        "lon": 53.48,
        "type": "Oil",
        "subtype": "Offshore",
        "notes": "Large Abu Dhabi offshore oil field",
        "capacity_scale": 90,
        "metric_text": "Oil capacity: 750,000 bbl/d"
    },
    {
        "name": "Lower Zakum",
        "lat": 24.73,
        "lon": 53.68,
        "type": "Oil",
        "subtype": "Offshore",
        "notes": "Major offshore oil field",
        "capacity_scale": 75,
        "metric_text": "Oil capacity: 450,000 bbl/d"
    },
    {
        "name": "Bab",
        "lat": 23.6,
        "lon": 54.1,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Large Abu Dhabi onshore field",
        "capacity_scale": 70,
        "metric_text": "Oil capacity: 420,000 bbl/d"
    },
    {
        "name": "Bu Hasa",
        "lat": 23.55,
        "lon": 53.4,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Major onshore oil field",
        "capacity_scale": 65,
        "metric_text": "Oil capacity: 650,000 bbl/d"
    },
    {
        "name": "Shah Gas",
        "lat": 23.07,
        "lon": 53.73,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Sour gas development in Abu Dhabi",
        "capacity_scale": 55,
        "metric_text": "Gas capacity: 1.0 Bcf/d"
    },
    {
        "name": "Ruwais Refinery",
        "lat": 24.12,
        "lon": 52.73,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Large downstream hub in western Abu Dhabi",
        "capacity_scale": 75,
        "metric_text": "Refining capacity: 837,000 bbl/d"
    },
    {
        "name": "Jebel Ali Desalination",
        "lat": 24.99,
        "lon": 55.07,
        "type": "Desalination",
        "subtype": "MSF + RO",
        "notes": "Major Dubai water and power complex",
        "capacity_scale": 60,
        "metric_text": "Water capacity: 2.2 million m3/day"
    },
    {
        "name": "Taweelah Desalination",
        "lat": 24.78,
        "lon": 54.7,
        "type": "Desalination",
        "subtype": "RO",
        "notes": "Large Abu Dhabi desalination complex",
        "capacity_scale": 55,
        "metric_text": "Water capacity: 909,000 m3/day"
    },
    {
        "name": "Al Dhafra Solar",
        "lat": 24.4,
        "lon": 54.45,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Utility-scale solar PV project near Abu Dhabi",
        "capacity_scale": 65,
        "metric_text": "Solar capacity: 2,000 MW"
    },
    {
        "name": "MBR Solar Park",
        "lat": 24.77,
        "lon": 55.36,
        "type": "Solar",
        "subtype": "PV + CSP",
        "notes": "Mohammed bin Rashid Al Maktoum Solar Park",
        "capacity_scale": 60,
        "metric_text": "Solar target: 5,000 MW"
    }
]

df_resources_uae = pd.DataFrame(uae_resource_sites)

print(f"Total United Arab Emirates resource sites: {len(df_resources_uae)}\n")
print(df_resources_uae.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_uae.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 12: 3D resource bars - United Arab Emirates
render_country_3d_bars(
    df_resources_uae,
    country_name='United Arab Emirates',
    output_path='output/middle_east_3d_bars_uae.html',
    center=[24.2, 54.3],
    zoom=6,
)


In [ ]:
# Cell 13: Curated resource sites - Kuwait
kuwait_resource_sites = [
    {
        "name": "Burgan",
        "lat": 29.08,
        "lon": 47.97,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "One of the largest conventional oil fields globally",
        "capacity_scale": 100,
        "metric_text": "Oil reserves: 66-70 billion bbl"
    },
    {
        "name": "Raudhatain",
        "lat": 29.77,
        "lon": 47.71,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Northern Kuwait oil field",
        "capacity_scale": 55,
        "metric_text": "Oil capacity: 230,000 bbl/d"
    },
    {
        "name": "Sabriya",
        "lat": 29.84,
        "lon": 47.88,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Northern Kuwait field with Jurassic reservoirs nearby",
        "capacity_scale": 50,
        "metric_text": "Oil capacity: 120,000 bbl/d"
    },
    {
        "name": "Minagish",
        "lat": 29.07,
        "lon": 47.07,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "West Kuwait oil field",
        "capacity_scale": 45,
        "metric_text": "Oil capacity: 190,000 bbl/d"
    },
    {
        "name": "Umm Gudair",
        "lat": 28.97,
        "lon": 47.3,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "West Kuwait oil field",
        "capacity_scale": 40,
        "metric_text": "Oil capacity: 110,000 bbl/d"
    },
    {
        "name": "Jurassic Gas",
        "lat": 29.7,
        "lon": 47.85,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Northern Kuwait non-associated gas development",
        "capacity_scale": 38,
        "metric_text": "Gas capacity: 600 MMcf/d"
    },
    {
        "name": "Mina Al Ahmadi Refinery",
        "lat": 29.08,
        "lon": 48.13,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Major refining and export complex",
        "capacity_scale": 65,
        "metric_text": "Refining capacity: 466,000 bbl/d"
    },
    {
        "name": "Al Zour Refinery",
        "lat": 28.73,
        "lon": 48.27,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Large newer refining complex",
        "capacity_scale": 75,
        "metric_text": "Refining capacity: 615,000 bbl/d"
    },
    {
        "name": "Doha Desalination",
        "lat": 29.36,
        "lon": 47.8,
        "type": "Desalination",
        "subtype": "MSF",
        "notes": "Kuwait Bay desalination and power site",
        "capacity_scale": 35,
        "metric_text": "Water capacity: 250,000 m3/day"
    },
    {
        "name": "Shagaya Renewables",
        "lat": 29.2,
        "lon": 46.67,
        "type": "Solar",
        "subtype": "PV + CSP + Wind",
        "notes": "Renewable energy park in western Kuwait",
        "capacity_scale": 30,
        "metric_text": "Renewable capacity: 70 MW"
    }
]

df_resources_kuwait = pd.DataFrame(kuwait_resource_sites)

print(f"Total Kuwait resource sites: {len(df_resources_kuwait)}\n")
print(df_resources_kuwait.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_kuwait.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 14: 3D resource bars - Kuwait
render_country_3d_bars(
    df_resources_kuwait,
    country_name='Kuwait',
    output_path='output/middle_east_3d_bars_kuwait.html',
    center=[29.3, 47.6],
    zoom=7,
)


In [ ]:
# Cell 15: Curated resource sites - Qatar
qatar_resource_sites = [
    {
        "name": "North Field",
        "lat": 26.5,
        "lon": 51.9,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Qatar side of the worlds largest gas field",
        "capacity_scale": 100,
        "metric_text": "Gas reserves: 900 Tcf"
    },
    {
        "name": "Dukhan",
        "lat": 25.42,
        "lon": 50.78,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Historic onshore oil field",
        "capacity_scale": 45,
        "metric_text": "Oil capacity: 300,000 bbl/d"
    },
    {
        "name": "Al Shaheen",
        "lat": 26.58,
        "lon": 52.35,
        "type": "Oil",
        "subtype": "Offshore",
        "notes": "Large offshore oil field above the North Field",
        "capacity_scale": 60,
        "metric_text": "Oil capacity: 300,000 bbl/d"
    },
    {
        "name": "Idd El-Shargi",
        "lat": 26.55,
        "lon": 52.0,
        "type": "Oil",
        "subtype": "Offshore",
        "notes": "Offshore oil field east of Qatar",
        "capacity_scale": 40,
        "metric_text": "Oil capacity: 100,000 bbl/d"
    },
    {
        "name": "Bul Hanine",
        "lat": 25.6,
        "lon": 52.1,
        "type": "Oil",
        "subtype": "Offshore",
        "notes": "Offshore oil and gas field",
        "capacity_scale": 35,
        "metric_text": "Oil capacity: 45,000 bbl/d"
    },
    {
        "name": "Pearl GTL",
        "lat": 25.93,
        "lon": 51.56,
        "type": "Refinery",
        "subtype": "Gas-to-Liquids",
        "notes": "Large gas-to-liquids complex at Ras Laffan",
        "capacity_scale": 70,
        "metric_text": "GTL output: 140,000 bbl/d"
    },
    {
        "name": "Ras Laffan Refinery",
        "lat": 25.93,
        "lon": 51.55,
        "type": "Refinery",
        "subtype": "Condensate Refinery",
        "notes": "Ras Laffan condensate refining complex",
        "capacity_scale": 55,
        "metric_text": "Refining capacity: 292,000 bbl/d"
    },
    {
        "name": "Ras Abu Fontas Desal",
        "lat": 25.17,
        "lon": 51.56,
        "type": "Desalination",
        "subtype": "MSF + RO",
        "notes": "Major desalination and power complex near Doha",
        "capacity_scale": 45,
        "metric_text": "Water capacity: 160,000 m3/day"
    },
    {
        "name": "Al Kharsaah Solar",
        "lat": 25.43,
        "lon": 51.07,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Large utility-scale solar PV plant",
        "capacity_scale": 35,
        "metric_text": "Solar capacity: 800 MW"
    }
]

df_resources_qatar = pd.DataFrame(qatar_resource_sites)

print(f"Total Qatar resource sites: {len(df_resources_qatar)}\n")
print(df_resources_qatar.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_qatar.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 16: 3D resource bars - Qatar
render_country_3d_bars(
    df_resources_qatar,
    country_name='Qatar',
    output_path='output/middle_east_3d_bars_qatar.html',
    center=[25.3, 51.2],
    zoom=7,
)


In [ ]:
# Cell 17: Curated resource sites - Oman
oman_resource_sites = [
    {
        "name": "Fahud",
        "lat": 22.33,
        "lon": 56.48,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Major northern Oman oil field",
        "capacity_scale": 55,
        "metric_text": "Oil capacity: 100,000 bbl/d"
    },
    {
        "name": "Yibal",
        "lat": 22.19,
        "lon": 56.08,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Long-producing field in northern Oman",
        "capacity_scale": 45,
        "metric_text": "Oil capacity: 40,000 bbl/d"
    },
    {
        "name": "Marmul",
        "lat": 18.16,
        "lon": 55.18,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Southern Oman oil field",
        "capacity_scale": 42,
        "metric_text": "Oil capacity: 80,000 bbl/d"
    },
    {
        "name": "Mukhaizna",
        "lat": 19.52,
        "lon": 56.42,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Heavy oil field using enhanced recovery",
        "capacity_scale": 50,
        "metric_text": "Oil capacity: 120,000 bbl/d"
    },
    {
        "name": "Khazzan",
        "lat": 22.1,
        "lon": 56.8,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Tight gas development in Block 61",
        "capacity_scale": 65,
        "metric_text": "Gas capacity: 1.5 Bcf/d"
    },
    {
        "name": "Saih Rawl",
        "lat": 22.32,
        "lon": 56.76,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Gas field feeding LNG and domestic supply",
        "capacity_scale": 50,
        "metric_text": "Gas capacity: 500 MMcf/d"
    },
    {
        "name": "Sohar Refinery",
        "lat": 24.47,
        "lon": 56.63,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Major refinery and industrial port complex",
        "capacity_scale": 55,
        "metric_text": "Refining capacity: 198,000 bbl/d"
    },
    {
        "name": "Mina Al Fahal Refinery",
        "lat": 23.63,
        "lon": 58.52,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Muscat-area refinery",
        "capacity_scale": 35,
        "metric_text": "Refining capacity: 106,000 bbl/d"
    },
    {
        "name": "Ibri Solar",
        "lat": 23.27,
        "lon": 56.52,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Large utility-scale solar PV plant",
        "capacity_scale": 38,
        "metric_text": "Solar capacity: 500 MW"
    },
    {
        "name": "Sohar Aluminium",
        "lat": 24.48,
        "lon": 56.6,
        "type": "Refinery",
        "subtype": "Aluminium Smelter",
        "notes": "Aluminium smelting complex at Sohar",
        "capacity_scale": 40,
        "metric_text": "Aluminium output: 395,000 t/yr"
    }
]

df_resources_oman = pd.DataFrame(oman_resource_sites)

print(f"Total Oman resource sites: {len(df_resources_oman)}\n")
print(df_resources_oman.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_oman.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 18: 3D resource bars - Oman
render_country_3d_bars(
    df_resources_oman,
    country_name='Oman',
    output_path='output/middle_east_3d_bars_oman.html',
    center=[21.0, 57.0],
    zoom=5.4,
)


In [ ]:
# Cell 19: Curated resource sites - Bahrain
bahrain_resource_sites = [
    {
        "name": "Bahrain Field",
        "lat": 26.04,
        "lon": 50.55,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Awali oil field, Bahrain original producing field",
        "capacity_scale": 45,
        "metric_text": "Oil capacity: 45,000 bbl/d"
    },
    {
        "name": "Khaleej Al Bahrain",
        "lat": 26.45,
        "lon": 50.35,
        "type": "Oil",
        "subtype": "Offshore",
        "notes": "Offshore unconventional oil and gas discovery area",
        "capacity_scale": 35,
        "metric_text": "Resource estimate: 80 billion bbl oil in place"
    },
    {
        "name": "Tatweer Gas",
        "lat": 26.03,
        "lon": 50.58,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Associated and non-associated gas production operations",
        "capacity_scale": 35,
        "metric_text": "Gas output: 1.5 Bcf/d"
    },
    {
        "name": "Sitra Refinery",
        "lat": 26.15,
        "lon": 50.63,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Bahrain Petroleum Company refinery",
        "capacity_scale": 45,
        "metric_text": "Refining capacity: 267,000 bbl/d"
    },
    {
        "name": "Alba Smelter",
        "lat": 26.1,
        "lon": 50.57,
        "type": "Refinery",
        "subtype": "Aluminium Smelter",
        "notes": "Large aluminium smelting complex",
        "capacity_scale": 55,
        "metric_text": "Aluminium output: 1.6 million t/yr"
    },
    {
        "name": "Al Dur Desalination",
        "lat": 25.95,
        "lon": 50.61,
        "type": "Desalination",
        "subtype": "RO",
        "notes": "Water and power production complex",
        "capacity_scale": 35,
        "metric_text": "Water capacity: 218,000 m3/day"
    },
    {
        "name": "Hidd Desalination",
        "lat": 26.22,
        "lon": 50.66,
        "type": "Desalination",
        "subtype": "MSF",
        "notes": "Hidd power and water station",
        "capacity_scale": 30,
        "metric_text": "Water capacity: 409,000 m3/day"
    },
    {
        "name": "Askar Solar",
        "lat": 26.06,
        "lon": 50.62,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Representative solar deployment area in southern Bahrain",
        "capacity_scale": 18,
        "metric_text": "Solar capacity: 5 MW"
    }
]

df_resources_bahrain = pd.DataFrame(bahrain_resource_sites)

print(f"Total Bahrain resource sites: {len(df_resources_bahrain)}\n")
print(df_resources_bahrain.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_bahrain.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 20: 3D resource bars - Bahrain
render_country_3d_bars(
    df_resources_bahrain,
    country_name='Bahrain',
    output_path='output/middle_east_3d_bars_bahrain.html',
    center=[26.05, 50.55],
    zoom=9,
)


In [ ]:
# Cell 21: Curated resource sites - Yemen
yemen_resource_sites = [
    {
        "name": "Masila Block 14",
        "lat": 15.42,
        "lon": 49.02,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Hadramawt oil production area",
        "capacity_scale": 55,
        "metric_text": "Oil capacity: 120,000 bbl/d peak"
    },
    {
        "name": "Marib-Jawf Basin",
        "lat": 15.47,
        "lon": 45.32,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Central Yemen oil and gas producing basin",
        "capacity_scale": 50,
        "metric_text": "Oil capacity: 200,000 bbl/d peak"
    },
    {
        "name": "Shabwa Block S1",
        "lat": 14.8,
        "lon": 46.75,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Shabwa governorate oil block",
        "capacity_scale": 38,
        "metric_text": "Oil capacity: 50,000 bbl/d peak"
    },
    {
        "name": "Jannah Hunt",
        "lat": 15.23,
        "lon": 45.9,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Oil field area east of Marib",
        "capacity_scale": 35,
        "metric_text": "Oil capacity: 45,000 bbl/d peak"
    },
    {
        "name": "Marib Gas",
        "lat": 15.43,
        "lon": 45.33,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Gas resources feeding power and LNG value chain",
        "capacity_scale": 45,
        "metric_text": "Gas reserves: 10 Tcf"
    },
    {
        "name": "Balhaf LNG",
        "lat": 13.98,
        "lon": 48.18,
        "type": "Refinery",
        "subtype": "LNG Export",
        "notes": "LNG liquefaction and export terminal on the Gulf of Aden",
        "capacity_scale": 55,
        "metric_text": "LNG capacity: 6.7 million t/yr"
    },
    {
        "name": "Aden Refinery",
        "lat": 12.79,
        "lon": 44.99,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Aden coastal refinery complex",
        "capacity_scale": 35,
        "metric_text": "Refining capacity: 150,000 bbl/d"
    },
    {
        "name": "Marib Refinery",
        "lat": 15.46,
        "lon": 45.32,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Small refinery serving inland production",
        "capacity_scale": 22,
        "metric_text": "Refining capacity: 10,000 bbl/d"
    },
    {
        "name": "Al Jawf Solar",
        "lat": 16.6,
        "lon": 44.6,
        "type": "Solar",
        "subtype": "PV Potential",
        "notes": "Representative high-irradiance solar resource area",
        "capacity_scale": 20,
        "metric_text": "Solar resource: 2,200 kWh/m2/yr"
    }
]

df_resources_yemen = pd.DataFrame(yemen_resource_sites)

print(f"Total Yemen resource sites: {len(df_resources_yemen)}\n")
print(df_resources_yemen.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_yemen.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 22: 3D resource bars - Yemen
render_country_3d_bars(
    df_resources_yemen,
    country_name='Yemen',
    output_path='output/middle_east_3d_bars_yemen.html',
    center=[15.6, 47.5],
    zoom=5.4,
)


In [ ]:
# Cell 23: Curated resource sites - Jordan
jordan_resource_sites = [
    {
        "name": "Risha Gas",
        "lat": 32.35,
        "lon": 38.92,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Eastern Jordan gas field",
        "capacity_scale": 30,
        "metric_text": "Gas output: 16 MMcf/d"
    },
    {
        "name": "Hamza Oil Field",
        "lat": 31.52,
        "lon": 36.33,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Small producing oil field",
        "capacity_scale": 12,
        "metric_text": "Oil output: 1,000 bbl/d"
    },
    {
        "name": "Arab Potash",
        "lat": 31.25,
        "lon": 35.47,
        "type": "Mineral",
        "subtype": "Potash",
        "notes": "Dead Sea potash production complex",
        "capacity_scale": 65,
        "metric_text": "Potash output: 2.5 million t/yr"
    },
    {
        "name": "Jordan Bromine",
        "lat": 31.2,
        "lon": 35.45,
        "type": "Mineral",
        "subtype": "Bromine",
        "notes": "Dead Sea bromine production",
        "capacity_scale": 45,
        "metric_text": "Bromine output: 140,000 t/yr"
    },
    {
        "name": "Eshidiya Phosphate",
        "lat": 29.93,
        "lon": 36.08,
        "type": "Mineral",
        "subtype": "Phosphate",
        "notes": "Large phosphate mine in southern Jordan",
        "capacity_scale": 60,
        "metric_text": "Phosphate output: 6 million t/yr"
    },
    {
        "name": "Al-Abiad Phosphate",
        "lat": 31.58,
        "lon": 36.05,
        "type": "Mineral",
        "subtype": "Phosphate",
        "notes": "Central Jordan phosphate mining area",
        "capacity_scale": 40,
        "metric_text": "Phosphate output: 1 million t/yr"
    },
    {
        "name": "Attarat Oil Shale",
        "lat": 31.1,
        "lon": 36.28,
        "type": "Mineral",
        "subtype": "Oil Shale",
        "notes": "Oil shale power and resource development area",
        "capacity_scale": 45,
        "metric_text": "Power capacity: 470 MW"
    },
    {
        "name": "Ma an Solar",
        "lat": 30.2,
        "lon": 35.73,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Solar PV cluster in southern Jordan",
        "capacity_scale": 38,
        "metric_text": "Solar capacity: 160 MW"
    },
    {
        "name": "Quweira Solar",
        "lat": 29.8,
        "lon": 35.3,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Utility-scale solar project near Aqaba",
        "capacity_scale": 32,
        "metric_text": "Solar capacity: 103 MW"
    }
]

df_resources_jordan = pd.DataFrame(jordan_resource_sites)

print(f"Total Jordan resource sites: {len(df_resources_jordan)}\n")
print(df_resources_jordan.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_jordan.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 24: 3D resource bars - Jordan
render_country_3d_bars(
    df_resources_jordan,
    country_name='Jordan',
    output_path='output/middle_east_3d_bars_jordan.html',
    center=[31.2, 36.6],
    zoom=6,
)


In [ ]:
# Cell 25: Curated resource sites - Israel
israel_resource_sites = [
    {
        "name": "Leviathan",
        "lat": 33.1,
        "lon": 33.7,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Large offshore Mediterranean gas field",
        "capacity_scale": 80,
        "metric_text": "Gas reserves: 22 Tcf"
    },
    {
        "name": "Tamar",
        "lat": 32.9,
        "lon": 34.1,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Major offshore gas field west of Haifa",
        "capacity_scale": 65,
        "metric_text": "Gas reserves: 10 Tcf"
    },
    {
        "name": "Karish",
        "lat": 33.2,
        "lon": 34.0,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Eastern Mediterranean offshore gas field",
        "capacity_scale": 45,
        "metric_text": "Gas reserves: 1.4 Tcf"
    },
    {
        "name": "Tanin",
        "lat": 33.27,
        "lon": 34.12,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Offshore gas discovery north of Karish",
        "capacity_scale": 30,
        "metric_text": "Gas reserves: 1.2 Tcf"
    },
    {
        "name": "Dead Sea Works",
        "lat": 31.05,
        "lon": 35.38,
        "type": "Mineral",
        "subtype": "Potash",
        "notes": "Dead Sea potash and mineral extraction complex",
        "capacity_scale": 60,
        "metric_text": "Potash output: 4 million t/yr"
    },
    {
        "name": "Haifa Refinery",
        "lat": 32.8,
        "lon": 35.05,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Northern coastal refining complex",
        "capacity_scale": 45,
        "metric_text": "Refining capacity: 197,000 bbl/d"
    },
    {
        "name": "Ashdod Refinery",
        "lat": 31.83,
        "lon": 34.65,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Southern coastal refinery",
        "capacity_scale": 38,
        "metric_text": "Refining capacity: 100,000 bbl/d"
    },
    {
        "name": "Sorek Desalination",
        "lat": 31.93,
        "lon": 34.7,
        "type": "Desalination",
        "subtype": "RO",
        "notes": "Large reverse-osmosis desalination plant",
        "capacity_scale": 55,
        "metric_text": "Water capacity: 624,000 m3/day"
    },
    {
        "name": "Hadera Desalination",
        "lat": 32.47,
        "lon": 34.88,
        "type": "Desalination",
        "subtype": "RO",
        "notes": "Major Mediterranean desalination plant",
        "capacity_scale": 48,
        "metric_text": "Water capacity: 525,000 m3/day"
    },
    {
        "name": "Ashalim Solar",
        "lat": 30.96,
        "lon": 34.7,
        "type": "Solar",
        "subtype": "CSP + PV",
        "notes": "Negev solar power complex",
        "capacity_scale": 35,
        "metric_text": "Solar capacity: 250 MW"
    }
]

df_resources_israel = pd.DataFrame(israel_resource_sites)

print(f"Total Israel resource sites: {len(df_resources_israel)}\n")
print(df_resources_israel.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_israel.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 26: 3D resource bars - Israel
render_country_3d_bars(
    df_resources_israel,
    country_name='Israel',
    output_path='output/middle_east_3d_bars_israel.html',
    center=[31.5, 35.0],
    zoom=6.4,
)


In [ ]:
# Cell 27: Curated resource sites - Syria
syria_resource_sites = [
    {
        "name": "Al-Omar",
        "lat": 35.07,
        "lon": 40.92,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Large oil field in eastern Syria",
        "capacity_scale": 55,
        "metric_text": "Oil capacity: 80,000 bbl/d peak"
    },
    {
        "name": "Al-Tanak",
        "lat": 35.1,
        "lon": 40.45,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Deir ez-Zor oil field",
        "capacity_scale": 45,
        "metric_text": "Oil capacity: 40,000 bbl/d peak"
    },
    {
        "name": "Rmelan",
        "lat": 37.05,
        "lon": 41.95,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Northeastern oil production area",
        "capacity_scale": 45,
        "metric_text": "Oil capacity: 90,000 bbl/d peak"
    },
    {
        "name": "Jbessa",
        "lat": 36.4,
        "lon": 40.75,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Hasakah province oil field area",
        "capacity_scale": 35,
        "metric_text": "Oil capacity: 35,000 bbl/d peak"
    },
    {
        "name": "Al-Shaer Gas",
        "lat": 34.82,
        "lon": 37.5,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Central Syria gas field",
        "capacity_scale": 38,
        "metric_text": "Gas output: 3 million m3/day peak"
    },
    {
        "name": "Conoco Gas Plant",
        "lat": 35.23,
        "lon": 40.32,
        "type": "Gas",
        "subtype": "Processing",
        "notes": "Major gas processing plant near Deir ez-Zor",
        "capacity_scale": 42,
        "metric_text": "Gas processing: 450 MMcf/d"
    },
    {
        "name": "Banias Refinery",
        "lat": 35.18,
        "lon": 35.95,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Mediterranean coastal refinery",
        "capacity_scale": 35,
        "metric_text": "Refining capacity: 130,000 bbl/d"
    },
    {
        "name": "Homs Refinery",
        "lat": 34.75,
        "lon": 36.7,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Inland refinery near Homs",
        "capacity_scale": 32,
        "metric_text": "Refining capacity: 107,000 bbl/d"
    },
    {
        "name": "Palmyra Phosphate",
        "lat": 34.57,
        "lon": 38.28,
        "type": "Mineral",
        "subtype": "Phosphate",
        "notes": "Central Syria phosphate deposits",
        "capacity_scale": 38,
        "metric_text": "Phosphate output: 2 million t/yr"
    },
    {
        "name": "Khnifis Phosphate",
        "lat": 34.06,
        "lon": 37.05,
        "type": "Mineral",
        "subtype": "Phosphate",
        "notes": "Phosphate mining area south of Homs",
        "capacity_scale": 32,
        "metric_text": "Phosphate output: 1 million t/yr"
    }
]

df_resources_syria = pd.DataFrame(syria_resource_sites)

print(f"Total Syria resource sites: {len(df_resources_syria)}\n")
print(df_resources_syria.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_syria.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 28: 3D resource bars - Syria
render_country_3d_bars(
    df_resources_syria,
    country_name='Syria',
    output_path='output/middle_east_3d_bars_syria.html',
    center=[35.0, 38.5],
    zoom=5.6,
)


In [ ]:
# Cell 29: Curated resource sites - Lebanon
lebanon_resource_sites = [
    {
        "name": "Offshore Block 9",
        "lat": 33.05,
        "lon": 34.95,
        "type": "Gas",
        "subtype": "Offshore Exploration",
        "notes": "Southern offshore exploration block",
        "capacity_scale": 30,
        "metric_text": "Exploration block: no commercial output yet"
    },
    {
        "name": "Offshore Block 4",
        "lat": 34.1,
        "lon": 34.95,
        "type": "Gas",
        "subtype": "Offshore Exploration",
        "notes": "Central offshore exploration block",
        "capacity_scale": 24,
        "metric_text": "Exploration block: no commercial output yet"
    },
    {
        "name": "Tripoli Oil Installations",
        "lat": 34.45,
        "lon": 35.84,
        "type": "Refinery",
        "subtype": "Oil Terminal",
        "notes": "Northern coastal oil storage and terminal site",
        "capacity_scale": 25,
        "metric_text": "Oil storage: 250,000 m3"
    },
    {
        "name": "Zahrani Oil Terminal",
        "lat": 33.5,
        "lon": 35.34,
        "type": "Refinery",
        "subtype": "Oil Terminal",
        "notes": "Southern coastal oil storage and terminal site",
        "capacity_scale": 25,
        "metric_text": "Oil storage: 180,000 m3"
    },
    {
        "name": "Sibline Limestone",
        "lat": 33.58,
        "lon": 35.45,
        "type": "Mineral",
        "subtype": "Limestone",
        "notes": "Cement and limestone production area",
        "capacity_scale": 28,
        "metric_text": "Cement capacity: 1.2 million t/yr"
    },
    {
        "name": "Chekka Limestone",
        "lat": 34.33,
        "lon": 35.73,
        "type": "Mineral",
        "subtype": "Limestone",
        "notes": "Northern coast cement and quarrying area",
        "capacity_scale": 30,
        "metric_text": "Cement capacity: 2 million t/yr"
    },
    {
        "name": "Beirut River Solar",
        "lat": 33.89,
        "lon": 35.54,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Urban solar PV installation",
        "capacity_scale": 12,
        "metric_text": "Solar capacity: 1 MW"
    },
    {
        "name": "Zahrani Solar",
        "lat": 33.48,
        "lon": 35.34,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Representative solar generation site near Zahrani",
        "capacity_scale": 15,
        "metric_text": "Solar capacity: 12 MW"
    }
]

df_resources_lebanon = pd.DataFrame(lebanon_resource_sites)

print(f"Total Lebanon resource sites: {len(df_resources_lebanon)}\n")
print(df_resources_lebanon.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_lebanon.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 30: 3D resource bars - Lebanon
render_country_3d_bars(
    df_resources_lebanon,
    country_name='Lebanon',
    output_path='output/middle_east_3d_bars_lebanon.html',
    center=[33.9, 35.8],
    zoom=7.5,
)


In [ ]:
# Cell 31: Curated resource sites - Turkey
turkey_resource_sites = [
    {
        "name": "Sakarya Gas Field",
        "lat": 42.55,
        "lon": 31.3,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Black Sea offshore gas development",
        "capacity_scale": 70,
        "metric_text": "Gas reserves: 710 Bcm"
    },
    {
        "name": "Raman",
        "lat": 37.82,
        "lon": 41.08,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Historic southeastern Turkey oil field",
        "capacity_scale": 30,
        "metric_text": "Oil output: 7,000 bbl/d"
    },
    {
        "name": "Garzan",
        "lat": 38.1,
        "lon": 41.45,
        "type": "Oil",
        "subtype": "Onshore",
        "notes": "Batman-Siirt area oil field",
        "capacity_scale": 25,
        "metric_text": "Oil output: 5,000 bbl/d"
    },
    {
        "name": "Tuz Golu Gas Storage",
        "lat": 38.7,
        "lon": 33.35,
        "type": "Gas",
        "subtype": "Storage",
        "notes": "Large underground gas storage facility",
        "capacity_scale": 45,
        "metric_text": "Storage capacity: 1.2 Bcm"
    },
    {
        "name": "Izmit Refinery",
        "lat": 40.76,
        "lon": 29.76,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "TUPRAS refinery near Izmit",
        "capacity_scale": 60,
        "metric_text": "Refining capacity: 227,000 bbl/d"
    },
    {
        "name": "Izmir Refinery",
        "lat": 38.76,
        "lon": 26.93,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Aliaga refining complex",
        "capacity_scale": 55,
        "metric_text": "Refining capacity: 239,000 bbl/d"
    },
    {
        "name": "STAR Refinery",
        "lat": 38.8,
        "lon": 26.93,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Aliaga refinery complex",
        "capacity_scale": 55,
        "metric_text": "Refining capacity: 214,000 bbl/d"
    },
    {
        "name": "Kirka Boron",
        "lat": 39.3,
        "lon": 30.55,
        "type": "Mineral",
        "subtype": "Boron",
        "notes": "Major boron mineral production area",
        "capacity_scale": 55,
        "metric_text": "Boron product: 2.5 million t/yr"
    },
    {
        "name": "Bigadic Boron",
        "lat": 39.4,
        "lon": 28.13,
        "type": "Mineral",
        "subtype": "Boron",
        "notes": "Western Turkey boron mining district",
        "capacity_scale": 45,
        "metric_text": "Boron product: 1 million t/yr"
    },
    {
        "name": "Karapinar Solar",
        "lat": 37.7,
        "lon": 33.55,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Large solar PV plant in Konya province",
        "capacity_scale": 55,
        "metric_text": "Solar capacity: 1,350 MW"
    }
]

df_resources_turkey = pd.DataFrame(turkey_resource_sites)

print(f"Total Turkey resource sites: {len(df_resources_turkey)}\n")
print(df_resources_turkey.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_turkey.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 32: 3D resource bars - Turkey
render_country_3d_bars(
    df_resources_turkey,
    country_name='Turkey',
    output_path='output/middle_east_3d_bars_turkey.html',
    center=[39.0, 35.0],
    zoom=4.6,
)


In [ ]:
# Cell 33: Curated resource sites - Egypt
egypt_resource_sites = [
    {
        "name": "Zohr",
        "lat": 31.95,
        "lon": 32.3,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Large Mediterranean offshore gas field",
        "capacity_scale": 90,
        "metric_text": "Gas reserves: 30 Tcf"
    },
    {
        "name": "West Delta Deep Marine",
        "lat": 31.7,
        "lon": 30.8,
        "type": "Gas",
        "subtype": "Offshore",
        "notes": "Mediterranean offshore gas development",
        "capacity_scale": 65,
        "metric_text": "Gas output: 1.5 Bcf/d"
    },
    {
        "name": "Abu Madi",
        "lat": 31.28,
        "lon": 31.36,
        "type": "Gas",
        "subtype": "Onshore",
        "notes": "Nile Delta gas field area",
        "capacity_scale": 45,
        "metric_text": "Gas reserves: 10 Tcf"
    },
    {
        "name": "Belayim",
        "lat": 28.6,
        "lon": 33.2,
        "type": "Oil",
        "subtype": "Gulf of Suez",
        "notes": "Major Gulf of Suez oil field area",
        "capacity_scale": 60,
        "metric_text": "Oil output: 90,000 bbl/d"
    },
    {
        "name": "Ras Gharib",
        "lat": 28.36,
        "lon": 33.08,
        "type": "Oil",
        "subtype": "Gulf of Suez",
        "notes": "Historic oil production area",
        "capacity_scale": 45,
        "metric_text": "Oil output: 25,000 bbl/d"
    },
    {
        "name": "Sukari Gold Mine",
        "lat": 24.95,
        "lon": 34.72,
        "type": "Mineral",
        "subtype": "Gold",
        "notes": "Large modern gold mine in the Eastern Desert",
        "capacity_scale": 55,
        "metric_text": "Gold output: 450,000 oz/yr"
    },
    {
        "name": "Abu Tartur Phosphate",
        "lat": 25.5,
        "lon": 30.08,
        "type": "Mineral",
        "subtype": "Phosphate",
        "notes": "Western Desert phosphate deposit",
        "capacity_scale": 42,
        "metric_text": "Phosphate reserves: 1 billion t"
    },
    {
        "name": "Benban Solar Park",
        "lat": 24.45,
        "lon": 32.75,
        "type": "Solar",
        "subtype": "PV",
        "notes": "Large solar PV park near Aswan",
        "capacity_scale": 70,
        "metric_text": "Solar capacity: 1,650 MW"
    },
    {
        "name": "Suez Refinery",
        "lat": 29.97,
        "lon": 32.55,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Refining complex near the Suez Canal",
        "capacity_scale": 45,
        "metric_text": "Refining capacity: 68,000 bbl/d"
    },
    {
        "name": "MIDOR Refinery",
        "lat": 31.0,
        "lon": 29.8,
        "type": "Refinery",
        "subtype": "Oil Refinery",
        "notes": "Alexandria-area refining complex",
        "capacity_scale": 48,
        "metric_text": "Refining capacity: 160,000 bbl/d"
    },
    {
        "name": "Al Galala Desalination",
        "lat": 29.45,
        "lon": 32.35,
        "type": "Desalination",
        "subtype": "RO",
        "notes": "Representative Red Sea desalination site",
        "capacity_scale": 35,
        "metric_text": "Water capacity: 150,000 m3/day"
    }
]

df_resources_egypt = pd.DataFrame(egypt_resource_sites)

print(f"Total Egypt resource sites: {len(df_resources_egypt)}\n")
print(df_resources_egypt.groupby("type").size().to_string())
print("\nSubtypes:")
print(df_resources_egypt.groupby(["type", "subtype"]).size().to_string())


In [ ]:
# Cell 34: 3D resource bars - Egypt
render_country_3d_bars(
    df_resources_egypt,
    country_name='Egypt',
    output_path='output/middle_east_3d_bars_egypt.html',
    center=[26.8, 30.8],
    zoom=5,
)
